In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
%cd drive/MyDrive

/content/drive/MyDrive


In [4]:
import torch
import math
from seq2seq_models import preprocess_text, tokenize_data, pad_or_truncate, Vocab, encode, Embedding
from sequential_models import Linear, Tanh, BasicRNN, LSTMCell, GRUCell

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Text preprocessing

In [5]:
dataset_path = "fra.txt"
src, tgt = [], []

with open(dataset_path) as file_object:
  for i, line in enumerate(file_object):
    result = preprocess_text(line)
    tokenize_data(result, src, tgt)

src = [pad_or_truncate(s) for s in src]
tgt = [pad_or_truncate(t) for t in tgt]
tgt = [["<bos>"] + t for t in tgt]
eng_vocab = Vocab(src)
fr_vocab = Vocab(tgt)

lookup_fxn = lambda sentence, vocab: [encode(vocab, s) for s in sentence]
src_lookup = [lookup_fxn(s, eng_vocab) for s in src]
tgt_lookup = [lookup_fxn(t, fr_vocab) for t in tgt]

src_lookup = torch.tensor(src_lookup, dtype=torch.int32)
tgt_lookup = torch.tensor(tgt_lookup, dtype=torch.int32)

target_label = tgt_lookup[:, 1:]
decoder_input = tgt_lookup[:, :-1]

label_valid_len = (target_label != fr_vocab["<pad>"]).type(torch.int32).sum(1)
decoder_valid_len = (decoder_input != fr_vocab["<pad>"]).type(torch.int32).sum(1)
src_valid_len = (src_lookup != eng_vocab["<pad>"]).type(torch.int32).sum(1)

In [6]:
def masked_softmax(X, valid_lens):
  """
  Mask <pad> tokens
  """
  def _mask(X, valid_len, value=0):
    maxlen = X.shape[-1]
    mask = torch.arange(maxlen) < valid_len.unsqueeze(-1)
    X[~mask] = value
    return X

  if valid_lens is None:
    return torch.nn.functional.softmax(X, dim=-1)
  else:
    if valid_lens.dim() == 1:
      valid_lens = valid_lens
    else:
      #flatten
      valid_lens = valid_lens.reshape(-1)

    X = _mask(X.reshape(-1, X.shape[-1]), valid_lens, value=-1e6)
    return torch.nn.functional.softmax(X, dim=-1)


In [8]:
class RNNEncoder():
  def __init__(self,
               vocab_size, #number of unique tokens
               feature_size, #number of embedding features
               n_neurons, #number of neurons in the RNN layers
               ):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.neurons = n_neurons

    self.embedding = Embedding(vocab_size, feature_size)

    self.rnn1 = BasicRNN(feature_size, n_neurons)


    #look at hidden state initialization
    self.rnn2 = BasicRNN(n_neurons, n_neurons)


  def __call__(self, x):
    h1 = torch.zeros((x.shape[0], self.neurons))

    dense_input = self.embedding(x)
    # [batch_size, seq_len, feature_size]
    output1, h1 = self.rnn1(dense_input, h1)
    # [seq_len, batch_size, n_hidden], [batch_size, n_hidden]
    output1 = output1.permute(1, 0, 2)
    h2 = torch.zeros((output1.shape[0], self.neurons))
    # [batch_size, seq_len, n_hidden], [batch_size, n_hidden]
    output2, h2 = self.rnn2(output1, h2)
    #[num_of_layers, batch_size, n_hidden], [seq_len, batch_size, n_hidden]
    return (h1, h2), output2

  def __repr__(self):
    rep = f"Encoder(\nEmbedding={self.embedding.embedding.shape},\nRNN=({self.feature_size, self.neurons}), \nRNN=({self.neurons, self.neurons})\n)"
    return rep

  def parameters(self):
    params = self.embedding.parameters() + self.rnn1.parameters() + self.rnn2.parameters()
    return params

In [42]:
class AdditiveAttention():
  def __init__(self, key_hidden_state, query_hidden_state, new_hidden_state):
    #shape of keys and values = [seq_len, batch_size, n_hidden]
    #hidden state of decoder(query) = [batch_size, n_hidden]
    self.Wk = Linear(key_hidden_state, new_hidden_state)
    self.Wq = Linear(query_hidden_state, new_hidden_state)
    self.Wv = Linear(new_hidden_state, 1)
    self.tanh = Tanh()

  def __repr__(self):
    return f"AdditiveAttention()"

  def __call__(self, keys, queries, values, valid_lens):
    features = self.Wk(keys) + self.Wq(queries)
    #features = [seq_len, batch_size, new_hidden_state]
    scores = self.Wv(self.tanh(features))
    #scores = [seq_len, batch_size, 1]
    scores = scores.permute(1, 2, 0)
    #scores = [batch_size, 1, seq_len]
    values = values.permute(1, 0, 2)
    #values = [batch_size, seq_len, hidden_state]
    self.weights = masked_softmax(scores.squeeze(1), valid_lens)
    self.weights = self.weights.unsqueeze(1)
    #self.weights = [batch_size, 1, seq_len]

    return torch.bmm(self.weights, values)
    #return [batch_size, 1, new_hidden_state]

  def parameters(self):
    return self.Wk.parameters() + self.Wq.parameters() + self.Wv.parameters()

In [43]:
class RNNAttentionDecoder():
  def __init__(self, feature_size, vocab_size, n_neurons, new_hidden_state, encoder_neurons):
    self.n_neurons = n_neurons
    self.embeddings = Embedding(vocab_size, feature_size)
    self.feature_size = feature_size
    self.vocab_size = vocab_size
    self.rnn1 = BasicRNN(feature_size + encoder_neurons, n_neurons)
    self.rnn2 = BasicRNN(n_neurons, n_neurons)
    self.attention = None
    self.linear = Linear(n_neurons, vocab_size)
    self.new_hidden = new_hidden_state

  def __call__(self, x, h_x, encoder_ouput, x_valid_lens):
    h1, h2 = h_x
    if self.attention is None:
      self.attention = AdditiveAttention(
          encoder_ouput.shape[-1],
          h2.shape[-1],
          self.new_hidden,
      )
    outputs = []
    seq = x.shape[1] #seq_len
    for t in range(seq):
      context = self.attention(encoder_ouput, h2, encoder_ouput, x_valid_lens)

      dense_input = self.embeddings(x[:, t]).unsqueeze(1)

      input_x = torch.cat((dense_input, context), -1)


      output1, h1 = self.rnn1(input_x, h1)
      output1 = output1.permute(1, 0, 2)
      output2, h2 = self.rnn2(output1, h2)

      output2 = output2.squeeze(0)
      outputs.append(output2)

    outputs = torch.stack(outputs)
    outputs = outputs.permute(1, 0, 2)
    logits = self.linear(outputs)
    return logits



  def __repr__(self):
    rep = f"Bahdanau Decoder(\nEmbedding={self.embeddings.embedding.shape},\nBahdanauAttention(), \nRNN=({self.feature_size, self.n_neurons}), \nRNN=({self.n_neurons, self.n_neurons}), \nLinear({self.n_neurons, self.vocab_size}) \n)"
    return rep

  def parameters(self):
    params = self.embeddings.parameters() + self.rnn1.parameters() + self.rnn2.parameters() + self.attention.parameters() + self.linear.parameters()
    return params


In [54]:
batch_size = 32

#encoder parameters
encoder_vocab_size = len(eng_vocab)
encoder_feature_size = 20
encoder_n_neurons = 10

#decoder parameters
decoder_vocab_size = len(fr_vocab)
decoder_feature_size = 20
decoder_n_neurons = 10
decoder_new_hidden = 20

encoder = RNNEncoder(
    encoder_vocab_size,
    encoder_feature_size,
    encoder_n_neurons
)

decoder = RNNAttentionDecoder(
    decoder_feature_size,
    decoder_vocab_size,
    decoder_n_neurons,
    decoder_new_hidden,
    encoder_n_neurons,
)

decoder.attention = AdditiveAttention(
         encoder_n_neurons,
          decoder_n_neurons,
          decoder_new_hidden,
      )


In [55]:
import torch.nn.functional as F
epochs = 100
params = encoder.parameters() + decoder.parameters()

for p in params:
  p.requires_grad = True

for epoch in range(epochs):
  idxs = torch.randint(0, batch_size + 1, (batch_size, ))
  #training data
  src_data = src_lookup[idxs]
  #input to decoder
  d_input = decoder_input[idxs]
  #target labels
  labels = target_label[idxs]

  #valid_lengths
  src_lens = src_valid_len[idxs]

  (h1, h2), encoder_output = encoder(src_data)
  #print(logits)
  logits = decoder(d_input, (h1, h2), encoder_output, src_lens)
  b, s, v = logits.shape
  logits = logits.reshape(b*s, -1)
  #use .long() on target values else cross_entropy will not work
  outs = labels.reshape(b*s).long()
  loss = F.cross_entropy(logits, outs)

  for p in params:
    p.grad = None

  loss.backward()

  lr = 0.101

  for p in params:
    p.data += -lr*p.grad

  print(loss.item())






13.545560836791992
13.493024826049805
14.0053071975708
13.427849769592285
13.231016159057617
12.52965259552002
11.894972801208496
11.406222343444824
10.920242309570312
10.659463882446289
10.097806930541992
9.652440071105957
9.011439323425293
8.597564697265625
8.016613960266113
7.359786510467529
6.935742378234863
6.251981735229492
5.7560930252075195
5.122897148132324
4.639370441436768
4.463770866394043
4.025067329406738
3.954099178314209
3.6253092288970947
3.935213565826416
3.8697516918182373
3.6063637733459473
3.5067970752716064
3.543754816055298
3.555168628692627
3.3661770820617676
3.6695973873138428
3.140531063079834
3.4842305183410645
3.1867120265960693
3.2098922729492188
3.270289659500122
3.234219551086426
3.1334173679351807
3.039872407913208
3.187739849090576
3.166537284851074
3.120631456375122
2.798490047454834
3.11217999458313
3.119365930557251
3.1180713176727295
2.8007311820983887
3.0416672229766846
3.0336484909057617
2.7499120235443115
2.892122983932495
3.152742624282837
2.925

In [ ]:
def transpose

In [7]:
class DotProductAttention():
  def __init__(self):
    pass

  def __repr__(self):
    return f"DotProductAttention()"

  def __call__(self, queries, keys, values, valid_lens=None):
    d = queries.shape[-1]
    scores = torch.bmm(queries, keys.transpose(1,2))/math.sqrt(d)
    weights = masked_softmax(scores, valid_lens)
    return torch.bmm(weights, values)

In [ ]:
class MultiHeadAttention():
  def __init__(self, query_size, key_size, value_size, new_hidden, num_heads):
    self.Wq = Linear(query_size, new_hidden)
    self.Wk = Linear(key_size, new_hidden)
    self.Wv = Linear(value_size, new_hidden)
    self.Wo = Linear(new_hidden, new_hidden)
    self.heads = num_heads
    self.attention = DotProductAttention()

  def transpose_qkv(self, X):
    batch_size, seq_len, hidden_size = X.shape
    #[batch_size, seq_len, num_heads, new_hidden/num_heads]
    X = X.reshape(batch_size, seq_len, self.heads, -1)
    #[batch_size, num_heads, seq_len, new_hidden/num_heads]
    X = X.permute(0, 2, 1, 3)
    #[batch_size*num_heads, seq_len, new_hidden/num_heads]
    X = X.reshape(-1, seq_len, X.shape[3])
    return X

  def transpose_output(self, X):
    X = X.reshape(-1, self.heads, X.shape[2], X.shape[3])
    X = X.permute(0, 2, 1, 3)
    X = X.reshape(X.shape[0], X.shape[1], -1)
    return X


  def __call__(self, keys, values, queries, valid_lens):
    queries = self.transpose_qkv(self.Wq(queries))
    keys = self.transpose_qkv(self.Wk(keys))
    values = self.trnaspose_qkv(self.Wv(values))
    #look at this side
    if valid_lens is not None:
      valid_lens = torch.repeat_interleave(valid_lens, self.heads)

    outputs = self.attention(queries, keys, values, valid_lens)
    outputs = self.transpose_output(outputs)
    return self.Wo(outputs)

  def __repr__(self):
    pass

  def parameters(self):
    pass